설치 검증 셀

In [2]:
import shutil
import sys

print("Python:", sys.version.split()[0])
print("가상환경 경로:", sys.prefix)
print("가상환경 안에서 실행 중?", ".venv" in sys.prefix)
print()

# OS 패키지 확인 (pip로 설치되지 않는 것들)
print("Poppler  :", "OK" if shutil.which("pdftoppm") else "✗ 미설치 — Path 확인 후 재시작")
print("Tesseract:", "OK" if shutil.which("tesseract") else "✗ 미설치 — Path 확인 후 재시작")
print()

# 파이썬 패키지 확인
for mod in ["langchain", "unstructured", "chromadb", "tiktoken", "PIL"]:
    try:
        __import__(mod)
        print(f"{mod:14s}: OK")
    except ImportError:
        print(f"{mod:14s}: ✗ 미설치")

Python: 3.11.9
가상환경 경로: c:\workspaces\multimodal-rag\.venv
가상환경 안에서 실행 중? True

Poppler  : OK
Tesseract: OK

langchain     : OK
unstructured  : OK
chromadb      : OK
tiktoken      : OK
PIL           : OK


추출 함수

In [3]:
# PDF에서 표, 텍스트, 이미지를 추출하고 텍스트를 분할하는 도구 불러오기
from langchain_text_splitters import CharacterTextSplitter
from unstructured.partition.pdf import partition_pdf


def extract_pdf_elements(path, fname):
    """
    PDF 파일에서 이미지, 테이블, 텍스트 블록을 추출합니다.
    path: 파일 경로 (추출된 이미지 파일이 저장될 위치)
    fname: 파일 이름
    """
    return partition_pdf(
        filename=path + fname,
        strategy="hi_res",                # 레이아웃 분석 모델로 정밀 추출 (교재에 없음! 필수 추가)
        extract_image_block_types=["Image"],                       # 추출할 요소 종류: 이미지
        extract_image_block_output_dir=os.path.join(path, "figures"),  # 저장 위치: data/figures
        infer_table_structure=True,       # 테이블 구조를 추론
        chunking_strategy="by_title",     # 타이틀을 기준으로 텍스트를 블록으로 분할
        max_characters=4000,              # 최대 4000자로 텍스트 블록을 제한
        new_after_n_chars=3800,           # 3800자 이후에 새로운 블록 생성
        combine_text_under_n_chars=2000,  # 2000자 이하의 텍스트는 결합
    )

추출한 요소를 종류별로 분류

In [4]:
def categorize_elements(raw_pdf_elements):
    """
    PDF에서 추출한 요소들을 테이블과 텍스트로 분류합니다.
    raw_pdf_elements: unstructured.documents.elements 리스트
    """
    tables = []
    texts = []
    for element in raw_pdf_elements:
        if "unstructured.documents.elements.Table" in str(type(element)):
            tables.append(str(element))   # 테이블 요소를 저장
        elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
            texts.append(str(element))    # 텍스트 요소를 저장
    return texts, tables

경로 설정 및 실행

In [5]:
# os 시스템 패키지
import os

# 현재 작업 디렉터리 경로를 반환합니다.
current_directory = os.getcwd()

# fname: 처리할 PDF 파일 이름 / fpath: PDF가 저장된 폴더 경로
fname = "economy_FFF.pdf"
fpath = os.path.join(os.path.dirname(current_directory), "data/")

print("현재 스크립트의 위치:", current_directory)
print("pdf 위치:", fpath)

# 실행하면 2~3분 걸립니다. 멈춘 게 아니니 기다리세요.
raw_pdf_elements = extract_pdf_elements(fpath, fname)
texts, tables = categorize_elements(raw_pdf_elements)

Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1


현재 스크립트의 위치: c:\workspaces\multimodal-rag\notebooks
pdf 위치: c:\workspaces\multimodal-rag\data/


No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 2325.70it/s]


텍스트 분할

In [6]:
# chunk_size=2000: 각 텍스트 블록의 최대 크기
# chunk_overlap=200: 청크 간 200자의 겹침을 허용해 중요한 정보를 잃지 않도록 함
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=2000, chunk_overlap=200
)

# 텍스트 리스트를 하나의 긴 문자열로 결합한 뒤 2000자 단위로 분할
joined_texts = " ".join(texts)
texts_2k_token = text_splitter.split_text(joined_texts)

print("분할 후 블록 수:", len(texts_2k_token))
print("원본 블록 수:", len(texts))
print(texts[0])
print(type(texts_2k_token))

분할 후 블록 수: 18
원본 블록 수: 12
2026년 10월 22일 공보 2026-10-17호

Hote

이 자료는 배포시부터 취급하여 주십시오.

제 목 : 경제상황 평가(2026년 10월)

□ 국내경제는 중동지역 불확실성 확대에도 반도체 경기의 강한 호조세와 여타부문으로의 파급 확산 등으로 기존 전망경로를 상회하는 높은 성장세 전망

□ 물가상승률은 누적된 비용압력이 전이되고 수요압력도 점차 확대됨에 따라 높은 수준 지속 예상

□ 전망경로에는 반도체 경기, 중동상황 등과 관련한 불확실성이 큰 상황

※ 자세한 내용은 <붙 임> 경제상황 평가(2026년 10월) 참조

[문의처 : 조사국 · 경제모형실]
<class 'list'>


API 키 불러오기

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OpenAI API 키가 없습니다. 한번 더 확인 부탁드립니다.")

요약 체인 구성

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama.llms import OllamaLLM


def generate_text_summaries(texts, tables, summarize_texts=False):
    """
    텍스트 및 표 데이터를 요약하여 검색에 활용할 수 있는 요약본을 생성합니다.
    texts: 텍스트 리스트
    tables: 표 리스트
    summarize_texts: 텍스트 요약을 활성화할지 여부를 결정하는 불리언 값
    """

    # Prompt 한국어 버전
    prompt_text_kor = """당신은 표와 텍스트를 요약하여 검색에 활용할 수 있도록 돕는 도우미입니다.
    이 요약본들은 임베딩되어 원본 텍스트나 표 요소를 검색하는 데 사용될 것입니다.
    주어진 표나 텍스트의 내용을 검색에 최적화된 간결한 요약으로 작성해주세요.
    요약할 표 또는 텍스트: {element}"""

    prompt = ChatPromptTemplate.from_template(prompt_text_kor)

    # 모델: GPT-4o-mini or Llama3.1 모델 사용
    # llamaModel = OllamaLLM(model="llama3.1:8b")
    model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

    # 요약 체인 = 프롬프트 + LLM + 출력 파서
    summarize_chain = {"element": lambda x: x} | prompt | model | StrOutputParser()

    text_summaries = []
    table_summaries = []

    # 텍스트 요약을 활성화한 경우
    if texts and summarize_texts:
        text_summaries = summarize_chain.batch(texts, {"max_concurrency": 5})
    # 텍스트 요약을 사용하지 않는 경우
    elif texts:
        text_summaries = texts

    # 테이블 데이터 요약
    if tables:
        table_summaries = summarize_chain.batch(tables, {"max_concurrency": 5})

    return text_summaries, table_summaries


text_summaries, table_summaries = generate_text_summaries(
    texts_2k_token, tables, summarize_texts=True
)
print(text_summaries)
print(table_summaries)

"""max_concurrency=5는 한 번에 5개씩 동시에 요약하라는 뜻입니다. 숫자를 올리면 빨라지지만 API 요청 제한에 걸릴 수 있습니다. """

['**2026년 10월 경제상황 평가 요약**\n\n- **국내경제 전망**: 중동지역의 불확실성에도 불구하고 반도체 경기 호조와 다른 부문으로의 긍정적 파급 효과로 인해 높은 성장세가 예상됨.\n- **물가상승률**: 누적된 비용 압력과 확대되는 수요 압력으로 인해 물가상승률이 높은 수준을 지속할 것으로 보임.\n- **불확실성 요인**: 반도체 경기와 중동 상황 등 여러 불확실성이 전망 경로에 영향을 미칠 가능성이 있음.\n\n**세계경제 동향**:\n- 높은 에너지 가격에도 불구하고 AI 투자로 인해 세계경제는 양호한 성장세를 유지할 전망.\n- 세계교역은 지정학적 리스크와 통상 갈등으로 제약을 받지만, AI 인프라 투자 관련 교역은 증가할 것으로 예상됨.\n- 국제유가는 중동 상황에 따라 변동성이 크지만, 중장기적으로는 공급 증가로 하락할 가능성이 있음.\n- 글로벌 반도체 경기는 AI 관련 인프라 투자 덕분에 견조할 것으로 보이나, 수익화 및 투자 속도에 대한 불확실성이 존재함.', '**ZHH PMI 요약**\n\n1. **국내경제 전망**:\n   - 중동지역 불확실성에도 불구하고 반도체 경기 호조로 높은 성장세 지속.\n   - 3/4분기 IT부문 수출 및 투자 호조로 성장률 유지.\n   - 4/4분기 내수 흐름 양호, 비IT 업황 개선 예상.\n   - 내년에도 반도체 경기 확장세 지속, 높은 성장세 전망.\n   - 성장률: 올해 3.3%, 내년 2.9%를 상회할 것으로 예상.\n   - 불확실성 요인: 중동 사태, AI 투자, 반도체 경기, 금융 여건 변화.\n\n2. **소비자물가**:\n   - 소비자물가 상승률, 비용 및 수요 압력으로 목표수준 초과 예상.\n   - 3/4분기 소비자물가 2.9% 상승, 근원물가는 2.9%로 확대.\n   - 올해 소비자물가 상승률은 2.7%, 근원물가는 2.5% 예상, 내년에는 각각 2.3%, 2.5%를 초과할 것으로 전망.', '**요약:**\n\n- **소비자물가 및 근원물가**: 2023년 

'max_concurrency=5는 한 번에 5개씩 동시에 요약하라는 뜻입니다. 숫자를 올리면 빨라지지만 API 요청 제한에 걸릴 수 있습니다. '